# Day 10 lab: Capstone and checkpoint rehearsal

**Goal:** Build one end-to-end pipeline that touches every checklist section, explain every decision Spark made inside it, and answer any checklist question in under 2 minutes.

**Time:** about 2 h 45 min. Checklist: All sections.

Put [`lab_utils.py`](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/lab_utils.py) next to this notebook, run the setup cell, then work through the tasks in order. Write your code in the empty cell under each task and your observations in the notes cell. Keep the Spark UI open (the setup cell prints its address).

In [ ]:
from lab_utils import *

spark = get_spark()

## 🏗 Capstone pipeline

Structure it like a real project: one function per layer in a module, plus a pytest file as in the official Testing PySpark guide. For every step save explain("formatted") into your Lab notes and annotate it: join strategy, number of shuffles, pushdown and pruning, spill, and what AQE changed.

### Task 1. Raw data — 15 min

Write three CSV drops of 500,000 orders each with `make_orders(..., hot_share=0.2, null_share=0.02, seed=1)`, then seeds 2 and 3, adding about 1 % duplicate rows to each (`union` of a `sample`). Write `make_customers` and `make_products` as Parquet dimensions.

In [ ]:
# Task 1: your code here


*Task 1 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 2. Bronze — 20 min

Read the CSV drops with an explicit DDL schema (no inferSchema) and append them to a Delta bronze table, adding `F.input_file_name()` and a load timestamp as lineage columns. Proves 2.1, 7.3, 6.6.

[Chapter 7: Load and Behold](https://spark.apache.org/docs/latest/api/python/user_guide/loadandbehold.html)

In [ ]:
# Task 2: your code here


*Task 2 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 3. Silver with MERGE — 30 min

Clean with built-in functions only (trim and upper-case status, cast types, send rows with null keys to a reject table), deduplicate by order_id keeping the latest `order_ts` (row_number window), and MERGE into a silver Delta table so that loading the same drop twice changes nothing. Proves 6.1, 7.7.

[Delta merge docs](https://docs.delta.io/latest/delta-update.html#upsert-into-a-table-using-merge) · [Testing PySpark](https://spark.apache.org/docs/latest/api/python/getting_started/testing_pyspark.html)

In [ ]:
# Task 3: your code here


*Task 3 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 4. Gold with the right joins — 25 min

Join silver with customers and products and aggregate revenue per country, category and day. Predict the plan first, then check it: both dimensions broadcast, one shuffle for the aggregation, pushdown on the silver scan. Proves 3.2, 3.9, 6.4.

[Chapter 6: SQL](https://spark.apache.org/docs/latest/api/python/user_guide/sql.html)

In [ ]:
# Task 4: your code here


*Task 4 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 5. Find and fix the skew — 25 min

Force a sort-merge join with customers (broadcast off) and find the hot customer in the Stages tab (max vs median). Fix it twice, with AQE skew handling and with a broadcast, and record the numbers. Proves 4.2, 4.6, 4.4.

In [ ]:
# Task 5: your code here


*Task 5 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 6. Maintain the table — 20 min

`OPTIMIZE silver ZORDER BY (customer_id)`, then check the file count, the history entry and `VACUUM silver DRY RUN`. Proves 7.4, 7.5, 7.9.

[Delta optimizations](https://docs.delta.io/latest/optimizations-oss.html)

In [ ]:
# Task 6: your code here


*Task 6 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 7. Incremental gold from the change feed — 30 min

Enable CDF on silver, load a fourth drop through the MERGE, read only the changes since the last processed version, and update the gold totals from them (inserts and post-images add, pre-images and deletes subtract). Store the last processed version in a small Delta table. Proves 7.10.

[change_data_feed.py](https://github.com/delta-io/delta/blob/v4.4.1/examples/python/change_data_feed.py)

In [ ]:
# Task 7: your code here


*Task 7 notes: what you predicted, what the plan and the Spark UI showed.*